# Laboratorio AWS — Predicción de temperatura de un transformador

## Notebook 03 — Entrenamiento administrado con Amazon SageMaker SDK v3

**Objetivo:** ejecutar un Training Job administrado por Amazon SageMaker utilizando los datos de entrenamiento y prueba almacenados en Amazon S3.

**Flujo:** S3 → `ModelTrainer` → Training Job → artefacto del modelo → S3

### Versión del SDK
Este notebook está preparado para **SageMaker Python SDK v3**. En esta versión se utiliza `ModelTrainer` en lugar de las clases específicas de SDK v2 como `SKLearn` o `Estimator`.

### Control de consumo
- 1 instancia de entrenamiento.
- `ml.m5.large`.
- Dataset de 1.000 registros.
- Random Forest pequeño.
- Tiempo máximo configurado de 10 minutos.

> Las celdas de validación y preparación no crean Training Jobs. El consumo de cómputo comienza únicamente cuando se ejecuta la celda **Lanzar Training Job**.

## 1. Librerías

Se utilizan `boto3` para verificar y organizar objetos en S3 y `ModelTrainer` para lanzar el entrenamiento administrado.

In [ ]:
import os
import tempfile
from pathlib import Path

import boto3
import pandas as pd

import sagemaker
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, InputData, StoppingCondition
from sagemaker.core.shapes import OutputDataConfig
   
print(f"SageMaker SDK: {sagemaker.__version__}")
print(f"Boto3: {boto3.__version__}")

## 2. Configuración del proyecto

El notebook reutiliza el bucket del laboratorio y la estructura creada en los notebooks anteriores.

In [ ]:
BUCKET = "siemens-ml-lab-aiarch-2026"
REGION = boto3.Session().region_name or "us-east-2"

RAW_KEY = "data/raw/transformadores.csv"
TRAIN_PREFIX = "data/train/"
TEST_PREFIX = "data/test/"
TRAIN_KEY = "data/train/train.csv"
TEST_KEY = "data/test/test.csv"
MODEL_PREFIX = "model/sagemaker/"

s3 = boto3.client("s3", region_name=REGION)

print(f"Región: {REGION}")
print(f"Bucket: {BUCKET}")

## 3. Verificar acceso al bucket

El acceso se realiza mediante el IAM Role asociado a SageMaker Studio. No se escriben credenciales AWS en el notebook.

In [ ]:
s3.head_bucket(Bucket=BUCKET)
print("OK - acceso al bucket confirmado.")

## 4. Verificar los datos de entrenamiento y prueba

El Notebook 02 ya debe haber generado:

```text
s3://siemens-ml-lab-aiarch-2026/data/train/train.csv
s3://siemens-ml-lab-aiarch-2026/data/test/test.csv
```

Si estos archivos no existen, no se lanza ningún recurso de SageMaker.

In [ ]:
for key in [TRAIN_KEY, TEST_KEY]:
    s3.head_object(Bucket=BUCKET, Key=key)
    print(f"OK - encontrado: s3://{BUCKET}/{key}")

## 5. Revisar una muestra de los datos

Esta celda descarga temporalmente los CSV y permite comprobar que el formato sea compatible con el script de entrenamiento.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    train_local = Path(tmpdir) / "train.csv"
    test_local = Path(tmpdir) / "test.csv"

    s3.download_file(BUCKET, TRAIN_KEY, str(train_local))
    s3.download_file(BUCKET, TEST_KEY, str(test_local))

    train_df = pd.read_csv(train_local)
    test_df = pd.read_csv(test_local)

display(train_df.head())
print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")

## 6. Crear el script de entrenamiento

El Training Job ejecutará este archivo dentro del contenedor de SageMaker.

El script:
1. Lee `train.csv` y `test.csv` desde los canales de entrada.
2. Entrena un `RandomForestRegressor`.
3. Calcula MAE, RMSE y R² sobre el conjunto de prueba.
4. Guarda `model.joblib` en `/opt/ml/model/`, ubicación que SageMaker empaqueta como artefacto del modelo.

In [ ]:
training_code = r'''import argparse
import json
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "temperatura_transformador"

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=10)
    args = parser.parse_args()

    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
    test_dir = os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test")
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")

    train = pd.read_csv(os.path.join(train_dir, "train.csv"))
    test = pd.read_csv(os.path.join(test_dir, "test.csv"))

    X_train = train.drop(columns=[TARGET])
    y_train = train[TARGET]
    X_test = test.drop(columns=[TARGET])
    y_test = test[TARGET]

    model = RandomForestRegressor(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
   
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    metrics = {"mae": mae, "rmse": rmse, "r2": r2}
    print(json.dumps(metrics))

    os.makedirs(model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(model_dir, "model.joblib"))

if __name__ == "__main__":
    main()
'''

script_path = Path("train.py")
script_path.write_text(training_code, encoding="utf-8")
print(f"Script creado: {script_path.resolve()}")

## 7. Validar sintaxis sin consumir SageMaker

La validación se ejecuta en el kernel actual. No crea ningún Training Job.

In [ ]:
compile(training_code, "train.py", "exec")
print("OK - sintaxis validada.")

## 8. Obtener la imagen de entrenamiento

Se obtiene dinámicamente la imagen de Scikit-learn para la región actual en lugar de escribir manualmente una URI de ECR. Esto hace que el notebook sea más portable.

In [ ]:
from sagemaker.core import image_uris

TRAINING_INSTANCE = "ml.m5.large"

training_image = image_uris.retrieve(
    framework="sklearn",
    region=REGION,
    version="1.2-1",
    py_version="py3",
    instance_type=TRAINING_INSTANCE,
    image_scope="training"
)

print(training_image)

## 9. Configurar el entrenamiento

`ModelTrainer` reemplaza a `SKLearn` y a los estimadores específicos del SDK v2. La configuración de cómputo utiliza una sola instancia.

In [ ]:
role = sagemaker.get_execution_role()

source_code = SourceCode(
    source_dir=".",
    entry_script="train.py",
    ignore_patterns=[
        ".git", ".gitignore", ".ipynb_checkpoints", "__pycache__",
        ".env", "data", "model", "output"
    ]
)

compute = Compute(
    instance_type=TRAINING_INSTANCE,
    instance_count=1
)

stopping_condition = StoppingCondition(max_runtime_in_seconds=600)

input_data_config = [
    InputData(
        channel_name="train",
        data_source=f"s3://{BUCKET}/{TRAIN_PREFIX}",
        content_type="text/csv"
    ),
    InputData(
        channel_name="test",
        data_source=f"s3://{BUCKET}/{TEST_PREFIX}",
        content_type="text/csv"
    )
]

output_data_config = OutputDataConfig(
    s3_output_path=f"s3://{BUCKET}/{MODEL_PREFIX}"
)

print("Configuración preparada correctamente.")

## 10. Crear el `ModelTrainer`

Esta celda solamente construye el objeto de configuración. Todavía no se crea un Training Job.

In [ ]:
trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    base_job_name="transformadores-sklearn-v3",
    source_code=source_code,
    compute=compute,
    stopping_condition=stopping_condition,
    input_data_config=input_data_config,
    output_data_config=output_data_config,
    hyperparameters={
        "n-estimators": 100,
        "max-depth": 10
    }
)

print("ModelTrainer creado correctamente.")

## 11. Lanzar Training Job

**ATENCIÓN:** esta es la celda que crea consumo de cómputo en SageMaker.

El entrenamiento es pequeño, pero el costo depende de la instancia y del tiempo que permanezca activo el Training Job. El límite configurado es de 600 segundos.

In [ ]:
training_job = trainer.train(wait=True, logs=True)

print("Training Job finalizado.")
print(f"Nombre del Training Job: {training_job.job_name}")

## 12. Consultar el artefacto del modelo

SageMaker guarda el artefacto producido por `/opt/ml/model/` en S3.

In [ ]:
training_job_name = training_job.job_name
print(f"Training Job: {training_job_name}")
print("Consulta el recurso TrainingJob para identificar el S3ModelArtifacts generado.")

sm = boto3.client("sagemaker", region_name=REGION)
details = sm.describe_training_job(TrainingJobName=training_job_name)
model_artifact = details["ModelArtifacts"]["S3ModelArtifacts"]

print(f"Artefacto del modelo: {model_artifact}")

## 13. Verificación final en S3

El objetivo es evidenciar que el resultado del Training Job quedó almacenado en S3.

In [ ]:
parsed = model_artifact.replace(f"s3://{BUCKET}/", "")
print(f"s3://{BUCKET}/{parsed}")

response = s3.list_objects_v2(Bucket=BUCKET, Prefix=MODEL_PREFIX)
for item in response.get("Contents", []):
    print(item["Key"])


## 14. Resumen del Notebook 03

Se completó el flujo administrado:

```text
S3/data/train + S3/data/test
            ↓
       ModelTrainer
            ↓
     SageMaker Training Job
            ↓
     RandomForestRegressor
            ↓
       model.joblib
            ↓
        S3/model/
```

El siguiente notebook utilizará este artefacto para construir una solución de inferencia con `ModelBuilder` y, de forma opcional, un endpoint Serverless.